In [2]:
from transformers import pipeline
import pandas as pd

# Load the scraped data
df = pd.read_csv("cleaned_bank_reviews.csv")

# Load sentiment pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Apply to reviews
def analyze_sentiment(text):
    result = sentiment_pipeline(text[:512])[0]
    label = result['label'].lower()
    score = result['score']
    return label, score
df[['sentiment_label', 'sentiment_score']] = df['review'].apply(lambda x: pd.Series(analyze_sentiment(x)))

Device set to use cpu


In [3]:
# By bank and rating
sentiment_summary = df.groupby(['bank', 'rating'])[['sentiment_score']].mean().reset_index()

In [4]:
# 3. Save cleaned data
df.to_csv("sentiment_summary", index=False)
print(f"sentiment_summary: {len(df)}")
print(df.columns)
print(df.head())

sentiment_summary: 1234
Index(['bank', 'review', 'rating', 'date', 'source', 'normalized_date',
       'sentiment_label', 'sentiment_score'],
      dtype='object')
  bank                                             review  rating        date  \
0  CBE                                           good app       5  2025-06-11   
1  CBE                         So bad now and hard to use       5  2025-06-09   
2  CBE  it is so amazing app. but, it is better to upd...       5  2025-06-09   
3  CBE                                         v.good app       4  2025-06-09   
4  CBE                                      very good app       1  2025-06-09   

        source normalized_date sentiment_label  sentiment_score  
0  Google Play      2025-06-11        positive         0.999849  
1  Google Play      2025-06-09        negative         0.999806  
2  Google Play      2025-06-09        positive         0.949643  
3  Google Play      2025-06-09        positive         0.995270  
4  Google Play     

In [5]:
import pandas as pd

# Assuming your DataFrame is called `df`
# First, create a unique `review_id` (if it doesn't exist)
df = df.reset_index(drop=True)
df['review_id'] = df.index + 1  # or use UUIDs for uniqueness

# Rename 'review' to 'review_text'
df = df.rename(columns={'review': 'review_text'})

# Reorder columns as required
df = df[['review_id', 'review_text', 'sentiment_label', 'sentiment_score', 'bank', 'rating']]


In [6]:
df.to_csv("sentiment_analysis_results.csv", index=False)
print(df.columns)
print(df.head())

Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'bank', 'rating'],
      dtype='object')
   review_id                                        review_text  \
0          1                                           good app   
1          2                         So bad now and hard to use   
2          3  it is so amazing app. but, it is better to upd...   
3          4                                         v.good app   
4          5                                      very good app   

  sentiment_label  sentiment_score bank  rating  
0        positive         0.999849  CBE       5  
1        negative         0.999806  CBE       5  
2        positive         0.949643  CBE       5  
3        positive         0.995270  CBE       4  
4        positive         0.999868  CBE       1  


In [7]:
import spacy
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(text)
    tokens = [token.lemma_.lower() for token in doc if not token.is_stop and token.is_alpha]
    return ' '.join(tokens)

df['clean_text'] = df['review_text'].apply(preprocess)
df.to_csv("with_clean_text.csv", index=False)


In [8]:
print(df.columns)
print(df.head())

Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'bank', 'rating', 'clean_text'],
      dtype='object')
   review_id                                        review_text  \
0          1                                           good app   
1          2                         So bad now and hard to use   
2          3  it is so amazing app. but, it is better to upd...   
3          4                                         v.good app   
4          5                                      very good app   

  sentiment_label  sentiment_score bank  rating  \
0        positive         0.999849  CBE       5   
1        negative         0.999806  CBE       5   
2        positive         0.949643  CBE       5   
3        positive         0.995270  CBE       4   
4        positive         0.999868  CBE       1   

                                          clean_text  
0                                           good app  
1                                       bad h